# 07. Evidence Analysis Foundry Component Build

This notebook builds the real Microsoft Foundry pieces of the Carrot Tuning system one component at a time. It follows the useful code-first pattern from the FrontierWeekHack claims example: Python constructs a narrowly scoped Foundry component, checks what Azure returned, and leaves a readable history.

The similarity ends at the safety boundary. The claims example creates tool-using agents that recommend insurance actions. This workbench will create one **no-tools evidence analyst** only after deterministic local gates approve a redacted packet. It will not mount the repository, call Kaggriculture, edit source, create a Knowledge index, or independently recommend a parameter value.

## Development envelope and stop points

Safe notebook work may inspect project metadata, construct configuration records, create SDK request objects, and render those facts. Cloud mutations remain separate cells with an exact target and a human approval immediately before them. A successful model deployment does not authorize agent creation; a successful agent creation does not authorize a model invocation.

- [ ] 1. Record and validate the proposed model-deployment configuration.
- [x] 2. Read-only Foundry discovery.
- [x] 3. Build and review the exact model-deployment request.
- [x] 4. Create and verify the model deployment.
- [ ] 5. Build and review the no-tools prompt-agent definition from source-controlled instructions.
- [ ] 6. After approval: create and verify the prompt agent.
- [ ] 7. After approval: invoke one redacted fixed fixture and inspect its trace/output gates.

Each unchecked item is an explicit review boundary, not permission to run the later cells.

## Handoff: what we are building, and what happens next

This notebook is the source-visible build history for the first remote component of the Carrot Tuning system. It is designed to let the owner—and a successor who has not seen the prior conversation—understand the target architecture, inspect each proposed cloud change before it happens, and see exactly which decisions still belong to a human.

### The component in one sentence

We are building one stateless, **no-tools Foundry prompt agent** that receives a redacted, deterministic-local-evaluator-approved evidence brief and returns a constrained explanation for human review. It is an evidence analyst, not a farmer, game runner, optimizer, policy editor, or experiment approver.

### Authority map

- The local Kaggriculture evaluator remains the truth owner for target binding, simulation, matched outcomes, clean audits, and immutable evidence packets.
- The local `analyst/` package remains the truth owner for packet eligibility, redacted-brief construction, and deterministic response gating.
- Foundry receives only the approved redacted brief. Its first prompt agent may explain packet-backed facts, cite them, state limits, and leave `pending_human_review`; it may not use tools or reach the repository.
- A deterministic local gate, not the model, decides whether a response is structurally eligible. The human alone approves a new experiment envelope or a source-policy change.

### What already exists

- Notebook 06 and the local `analyst/` package already establish the immutable request/response shape, fail-closed eligibility gate, redacted brief, constrained instruction policy, and deterministic output gate. Their fast tests are simulator-free.
- The Foundry account `foundry-trainer-resource` and project `foundry-trainer` are in East US 2, inside `rg-foundry-trainer-eastus2`. Redacted traces have a 30-day policy and the resource group has a $5 monthly alerting budget.
- No model deployment, prompt agent, tool, connector, Knowledge index, model call, or model-generated trace exists.
- Section 1 below is a local desired-state record only. It proposes `evidence-analyst-luna-v1`: `gpt-5.6-luna` version `2026-07-09`, `DataZoneStandard`, capacity unit `1` (1,000 TPM), default content filter, no spillover, and `NoAutoUpgrade`. It remains `pending_human_review`.

### How this notebook will proceed

This workbench deliberately separates five kinds of work: desired-state record, read-only discovery, exact request construction, side-effecting apply, and Azure readback verification. A success in one category never authorizes the next. The near-term sequence is:

- Record and locally validate the desired model deployment.
- Inspect the SDK, project, deployments, and prompt agents read-only.
- Construct and review the exact deployment request.
- Only after explicit approval, apply that request and read it back.
- Construct and review the no-tools prompt-agent definition from the versioned local instructions.
- Only after separate approval, create and read back that prompt agent.
- Only after another approval, invoke one fixed redacted fixture and inspect its deterministic gates and redacted trace.

Each future automated decision must retain `review_status: pending_human_review` and a visible unchecked `- [ ]` task until the owner records a disposition.

### FrontierWeekHack sample: useful pattern, not authority

The local `FrontierWeekHack/claims` repository is a read-only example of course-style Foundry client setup, create/list/readback patterns, and agent lifecycle code. It can help us make the Python workbench legible. It does **not** authorize us to copy its tool-using claims agents, action recommendations, connections, credentials, cleanup behavior, or deployment commands. We do not modify or run that sample.

### Success criteria for this notebook

The notebook will be successful when it contains a readable, reviewed, source-visible path from the local evidence boundary through a minimal Foundry prompt agent and a fixed-fixture evaluation/tracing demonstration—without silently expanding the agent's authority. It is not successful merely because an Azure resource exists or a model produces fluent text.

### Immediate successor task

Study the claims sample read-only, report the small set of SDK patterns that safely map to this project, then set up component 2 as a **non-mutating** notebook section. Add its explanatory Markdown and the smallest code framework needed to validate local configuration and prepare a read-only inspection. Do not run Azure, deploy a model, create an agent, invoke a model, use sample credentials, or alter the sample repository unless the owner gives a fresh explicit approval.

For full continuity, read `docs/next-agent-foundry-notebook-handoff.md`, `docs/foundry-transition-handoff.md`, `docs/foundry-provisioning-decision-record.md`, `README.md`, `docs/current-state-and-next-slice.md`, and the Foundry section of `QUESTION_LEDGER.md` before making a substantive change.

## 1. Record the proposed model deployment

This is a pure Python record of the exact first deployment—not a call to Azure. `capacity_units=1` is the Azure deployment value for 1,000 TPM. Data Zone Standard keeps inference processing in Microsoft's US data zone; it is pay-per-token, not provisioned throughput. `NoAutoUpgrade` makes any later model-version change a deliberate review rather than an invisible change to the evidence chain.

The later apply component must consume this reviewed record rather than retyping model facts into a portal or a shell command.

In [ ]:
from dataclasses import dataclass
import re


@dataclass(frozen=True)
class EvidenceAnalystDeploymentConfig:
    """A reviewed desired state; this class performs no cloud operation."""

    subscription_id: str
    resource_group: str
    account_name: str
    project_name: str
    deployment_name: str
    model_name: str
    model_version: str
    model_format: str
    sku_name: str
    capacity_units: int
    content_filter_policy: str
    version_upgrade_policy: str
    spillover_deployment_name: None
    review_status: str


def validate_deployment_config(config: EvidenceAnalystDeploymentConfig) -> EvidenceAnalystDeploymentConfig:
    """Refuse drift from the reviewed, low-cost first-deployment envelope."""
    # Naming is checked before any later code could turn this record into a request.
    if not re.fullmatch(r'[\w.-]{2,64}', config.deployment_name):
        raise ValueError('deployment name must use 2-64 letters, digits, dots, hyphens, or underscores')
    # Pin the model identity so a version change is visible and reviewable.
    if (config.model_name, config.model_version, config.model_format) != (
        'gpt-5.6-luna', '2026-07-09', 'OpenAI',
    ):
        raise ValueError('model identity differs from the reviewed analyst candidate')
    # One Data Zone unit is the deliberately small, pay-per-token envelope.
    if config.sku_name != 'DataZoneStandard' or config.capacity_units != 1:
        raise ValueError('the first deployment is one US Data Zone capacity unit (1,000 TPM)')
    if config.content_filter_policy != 'Microsoft.DefaultV2':
        raise ValueError('the reviewed default content-filter policy is required')
    # No spillover or silent version change is allowed in the first slice.
    if config.version_upgrade_policy != 'NoAutoUpgrade' or config.spillover_deployment_name is not None:
        raise ValueError('version changes and overflow routing remain explicit later decisions')
    if config.review_status != 'pending_human_review':
        raise ValueError('the deployment stays blocked until explicit human approval')
    # Returning the same frozen value preserves the validated desired state.
    return config


# This is a local proposal, not a portal readback or an Azure request.
EVIDENCE_ANALYST_DEPLOYMENT_CONFIG = EvidenceAnalystDeploymentConfig(
    subscription_id='d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930',
    resource_group='rg-foundry-trainer-eastus2',
    account_name='foundry-trainer-resource',
    project_name='foundry-trainer',
    deployment_name='evidence-analyst-luna-v1',
    model_name='gpt-5.6-luna',
    model_version='2026-07-09',
    model_format='OpenAI',
    sku_name='DataZoneStandard',
    capacity_units=1,
    content_filter_policy='Microsoft.DefaultV2',
    version_upgrade_policy='NoAutoUpgrade',
    spillover_deployment_name=None,
    review_status='pending_human_review',
)
# Validation is intentionally explicit before another cell consumes the record.
VALIDATED_DEPLOYMENT_CONFIG = validate_deployment_config(EVIDENCE_ANALYST_DEPLOYMENT_CONFIG)


In [ ]:
def render_deployment_configuration(config: EvidenceAnalystDeploymentConfig) -> None:
    """Presentation only: format an already-validated local configuration."""
    from IPython.display import Markdown, display

    # This renderer reads facts; it does not choose values or call Azure.
    display(Markdown(f"""### Proposed first model deployment

| Setting | Recorded value |
|:---|:---|
| Account / project | `{config.account_name}` / `{config.project_name}` |
| Resource group | `{config.resource_group}` |
| Deployment name | `{config.deployment_name}` |
| Model | `{config.model_name}` `{config.model_version}` |
| SKU / capacity | `{config.sku_name}` / `{config.capacity_units}` unit = **1,000 TPM** |
| Content filter / upgrades | `{config.content_filter_policy}` / `{config.version_upgrade_policy}` |
| Spillover | none |
| Azure action | **not performed** |
| Review state | **{config.review_status}** |

- [ ] Review this deployment configuration: approve, revise, or reject.

> This output describes an in-memory configuration only. It does not send an Azure request.
"""))


# The display is separate so it can be hidden or cleared without changing the record.
render_deployment_configuration(VALIDATED_DEPLOYMENT_CONFIG)


### Proposed first model deployment

| Setting | Recorded value |
|:---|:---|
| Account / project | `foundry-trainer-resource` / `foundry-trainer` |
| Resource group | `rg-foundry-trainer-eastus2` |
| Deployment name | `evidence-analyst-luna-v1` |
| Model | `gpt-5.6-luna` `2026-07-09` |
| SKU / capacity | `DataZoneStandard` / `1` unit = **1,000 TPM** |
| Content filter / upgrades | `Microsoft.DefaultV2` / `NoAutoUpgrade` |
| Spillover | none |
| Azure action | **not performed** |
| Review state | **pending_human_review** |

- [ ] Review this deployment configuration: approve, revise, or reject.

> This output describes an in-memory configuration only. It does not send an Azure request.


## Component 2 context

Component 2 is now complete for the approved read-only scope. The notebook recorded the local SDK prerequisite result, the exact request boundary, and the actual Foundry response: the configured project was reachable, with zero model deployments and zero prompt agents. No Azure mutation or model invocation occurred.

The pure request has now been constructed, validated, and presented. The owner approved it, and component 4 applied and verified the deployment. The next phase is step 5: define the no-tools prompt-agent request; no agent has been created or invoked.

### Claims-sample reference (context only)

The FrontierWeekHack claims sample is useful as a teaching example for three small patterns we may later adapt: keep Foundry client and project wiring explicit; keep create, list, and readback/verification as visibly separate operations; and keep the important configuration and lifecycle decisions in source-controlled Python rather than hiding them in a portal. These are patterns for making our work inspectable, not instructions to copy the sample's authority.

The following claims-system features do **not** transfer to this project: tool use, claims-action authority, insurance recommendations, repository or data-file access, connections, credentials or `.env` handling, cleanup commands, agent creation, deployment, and invocation. The first Carrot Tuning component remains a no-tools evidence analyst that receives only a redacted, locally validated evidence brief.

The discovery framework is now implemented as separate plan, prerequisite, request, execution, and presentation cells. The next agent must preserve the read-only result and review task; it must not treat zero resources as permission to construct or apply a deployment.

## 2. Read-only Foundry discovery

### 2.1 Define the discovery plan

This first implementation slice defines a **plan**, not a cloud reader. It turns the reviewed deployment target into an explicit list of facts that a later approved discovery cell could inspect. Keeping this plan pure lets us review the target and the operation boundary without installing packages, creating credentials, contacting Azure, or guessing what a successful response will look like.

The plan's input is the validated desired-state configuration from section 1. Its output is a frozen, source-visible description of the target endpoint and the three intended read-only observations: project identity/reachability, existing model deployments, and existing prompt agents. It deliberately excludes package installation, credential construction, create/update/delete operations, model invocation, and durable result storage.

The next review point is the plan itself. Only after it is accepted may we decide separately whether to inspect local package metadata, change the environment, or run one authenticated read through a notebook cell.

In [ ]:
from dataclasses import dataclass


# Frozen fields make the reviewable plan stable after it is built.
@dataclass(frozen=True)
class ReadOnlyDiscoveryPlan:
    """Pure description of a future read-only Foundry inspection."""

    account_name: str
    project_name: str
    project_endpoint: str
    observations: tuple[str, ...]
    allows_environment_change: bool
    allows_cloud_mutation: bool
    allows_model_invocation: bool
    stores_cloud_output: bool


def build_read_only_discovery_plan(
    config: EvidenceAnalystDeploymentConfig,
) -> ReadOnlyDiscoveryPlan:
    """Build a reviewable discovery plan without importing or calling an SDK."""
    # Reuse the section-1 validator instead of inventing a second target contract.
    validate_deployment_config(config)
    # This URL is a described target only; constructing it performs no network work.
    return ReadOnlyDiscoveryPlan(
        account_name=config.account_name,
        project_name=config.project_name,
        project_endpoint=(
            f"https://{config.account_name}.services.ai.azure.com/api/projects/"
            f"{config.project_name}"
        ),
        observations=(
            "project identity and read reachability",
            "existing model deployments",
            "existing prompt agents",
        ),
        allows_environment_change=False,
        allows_cloud_mutation=False,
        allows_model_invocation=False,
        stores_cloud_output=False,
    )


# The plan is intentionally dormant until its review boundary is accepted.
DISCOVERY_PLAN = build_read_only_discovery_plan(VALIDATED_DEPLOYMENT_CONFIG)


### 2.2 Presentation-only discovery-plan view

The next cell is presentation-only. If run, it will format the pure plan already built above. It does not inspect packages, authenticate, contact Azure, or invent a discovery result. The displayed checklist is a review of the plan's permissions, not evidence that the future read has happened.

In [ ]:
def render_read_only_discovery_plan(plan: ReadOnlyDiscoveryPlan) -> None:
    """Presentation only: summarize a pure discovery plan for review."""
    from IPython.display import Markdown, display

    # The renderer reports declared permissions, not observed Azure state.
    observations = "\n".join(f"- `{item}`" for item in plan.observations)
    display(Markdown(f"""### Read-only discovery plan

| Planned target | Value |
|:---|:---|
| Account / project | `{plan.account_name}` / `{plan.project_name}` |
| Endpoint described | `{plan.project_endpoint}` |
| Future observations |
{observations} |
| Environment change | **not allowed** |
| Cloud mutation | **not allowed** |
| Model invocation | **not allowed** |
| Cloud output storage | **not allowed** |

- [ ] Review this discovery plan before authorizing any local prerequisite or authenticated read.

> This is a description of an in-memory plan. It is not project inventory or cloud evidence.
"""))


# Clearing or hiding this cell does not change DISCOVERY_PLAN.
render_read_only_discovery_plan(DISCOVERY_PLAN)


### Read-only discovery plan

| Planned target | Value |
|:---|:---|
| Account / project | `foundry-trainer-resource` / `foundry-trainer` |
| Endpoint described | `https://foundry-trainer-resource.services.ai.azure.com/api/projects/foundry-trainer` |
| Future observations |
- `project identity and read reachability`
- `existing model deployments`
- `existing prompt agents` |
| Environment change | **not allowed** |
| Cloud mutation | **not allowed** |
| Model invocation | **not allowed** |
| Cloud output storage | **not allowed** |

- [ ] Review this discovery plan before authorizing any local prerequisite or authenticated read.

> This is a description of an in-memory plan. It is not project inventory or cloud evidence.


### 2.3 Define the local SDK prerequisite check

Before a future authenticated read can be attempted, we need to know whether this Python environment already has the reviewed client libraries. This cell defines a **local-only** check of distribution metadata and import-spec availability. It does not import an Azure client, construct a credential, access the network, install anything, or alter the environment.

A missing package is reported as a prerequisite for a later decision. It is not an error to repair automatically, and a passing check would not authorize the cloud reader.

**Competition note:** the official rules require a clear-purpose AI agent, a written description, examples/lessons, and a short demonstration video. They do not require Python, `azure-identity`, `DefaultAzureCredential`, or direct SDK authentication. This check supports an optional code-first inspection path; it is not required merely to submit or demonstrate the Foundry agent. See `references/foundry-competition/README.md`.

In [ ]:
from importlib import metadata, util


def inspect_local_sdk_prerequisites() -> dict[str, dict[str, object]]:
    """Report local Foundry SDK presence without importing or installing it."""
    required_packages = (
        ("azure-ai-projects", "azure.ai.projects"),
        ("azure-identity", "azure.identity"),
    )
    results: dict[str, dict[str, object]] = {}

    for distribution_name, module_name in required_packages:
        # Distribution metadata tells us what is installed; it makes no network request.
        try:
            distribution_version = metadata.version(distribution_name)
        except metadata.PackageNotFoundError:
            distribution_version = None

        # Finding a module spec avoids importing the Azure client or running its startup code.
        try:
            module_available = util.find_spec(module_name) is not None
        except ModuleNotFoundError:
            module_available = False

        results[distribution_name] = {
            "module": module_name,
            "distribution_version": distribution_version,
            "module_available": module_available,
        }

    return results


# This approved local-only call reads package metadata but performs no installation or network access.
SDK_PREREQUISITES = inspect_local_sdk_prerequisites()


### 2.4 Presentation-only prerequisite view

This presentation cell is also dormant. If the local check is explicitly run later, it will format the actual package metadata it returns. It will not report a cloud connection, authentication success, deployment, or agent.

In [ ]:
def render_local_sdk_prerequisites(prerequisites: dict[str, dict[str, object]]) -> None:
    """Presentation only: format an actual local prerequisite result."""
    from IPython.display import Markdown, display

    rows = []
    for package_name, details in prerequisites.items():
        version = details["distribution_version"] or "not installed"
        module_state = "available" if details["module_available"] else "not available"
        rows.append(f"| `{package_name}` | `{version}` | **{module_state}** |")
    table_rows = "\n".join(rows)

    # This renderer reports local facts only; it never recommends installation.
    display(Markdown(f"""### Local SDK prerequisite result

| Distribution | Version | Module |
|:---|:---|:---|
{table_rows}

- [ ] Review the local prerequisite result before considering any environment change or cloud read.

> This is local package metadata, not Foundry project evidence.
"""))


# This approved renderer formats the local result; it does not infer cloud state.
render_local_sdk_prerequisites(SDK_PREREQUISITES)


### Local SDK prerequisite result

| Distribution | Version | Module |
|:---|:---|:---|
| `azure-ai-projects` | `2.7.0` | **available** |
| `azure-identity` | `1.25.3` | **available** |

- [ ] Review the local prerequisite result before considering any environment change or cloud read.

> This is local package metadata, not Foundry project evidence.


### 2.5 Define the authenticated read-only request

The local prerequisite check tells us that the client libraries are present, but it does not prove that this notebook should contact Foundry. This next pure slice turns the reviewed discovery plan into an explicit request record. It names the exact project target and the only observations permitted: project reachability/identity, existing model deployments, and existing prompt agents.

The request record is still not a client, credential, or network call. It stays `pending_human_review` until the owner approves this exact authenticated read. A successful read would not authorize deployment, agent creation, invocation, or any other mutation.

In [ ]:
@dataclass(frozen=True)
class ReadOnlyCloudDiscoveryRequest:
    """Exact read boundary; constructing this record performs no cloud work."""

    account_name: str
    project_name: str
    project_endpoint: str
    operations: tuple[str, ...]
    prohibited_operations: tuple[str, ...]
    review_status: str


def build_read_only_cloud_discovery_request(
    plan: ReadOnlyDiscoveryPlan,
) -> ReadOnlyCloudDiscoveryRequest:
    """Convert the pure plan into a reviewable, non-mutating request record."""
    # The plan is the only source of target and operation facts.
    if plan.allows_environment_change or plan.allows_cloud_mutation or plan.allows_model_invocation:
        raise ValueError('discovery request cannot inherit a mutating permission')

    # These are observations, not commands to create, update, delete, or invoke.
    return ReadOnlyCloudDiscoveryRequest(
        account_name=plan.account_name,
        project_name=plan.project_name,
        project_endpoint=plan.project_endpoint,
        operations=(
            "read project identity/reachability",
            "list existing model deployments",
            "list existing prompt agents",
        ),
        prohibited_operations=(
            "install or change packages",
            "create, update, or delete Azure resources",
            "invoke a model or agent",
        ),
        review_status="pending_human_review",
    )


READ_ONLY_CLOUD_DISCOVERY_REQUEST = build_read_only_cloud_discovery_request(DISCOVERY_PLAN)


### 2.6 Presentation-only request review

This card makes the proposed authenticated read concrete enough to approve or reject. It reports the target and the allowed/prohibited operations from the request record. It does not claim that the project was reached or that any resource exists.

In [ ]:
def render_read_only_cloud_discovery_request(
    request: ReadOnlyCloudDiscoveryRequest,
) -> None:
    """Presentation only: format the proposed read boundary."""
    from IPython.display import Markdown, display

    allowed = "\n".join(f"- {operation}" for operation in request.operations)
    prohibited = "\n".join(f"- {operation}" for operation in request.prohibited_operations)
    # This output describes authorization scope; it is not a cloud response.
    display(Markdown(f"""### Proposed authenticated read-only inspection

| Target | Value |
|:---|:---|
| Account / project | `{request.account_name}` / `{request.project_name}` |
| Endpoint | `{request.project_endpoint}` |
| Review state | **{request.review_status}** |

**Allowed observations**
{allowed}

**Prohibited operations**
{prohibited}

- [ ] Approve or revise this exact authenticated read before any credential or client is constructed.

> No Azure request has been made by this cell.
"""))


render_read_only_cloud_discovery_request(READ_ONLY_CLOUD_DISCOVERY_REQUEST)


### Proposed authenticated read-only inspection

| Target | Value |
|:---|:---|
| Account / project | `foundry-trainer-resource` / `foundry-trainer` |
| Endpoint | `https://foundry-trainer-resource.services.ai.azure.com/api/projects/foundry-trainer` |
| Review state | **pending_human_review** |

**Allowed observations**
- read project identity/reachability
- list existing model deployments
- list existing prompt agents

**Prohibited operations**
- install or change packages
- create, update, or delete Azure resources
- invoke a model or agent

- [ ] Approve or revise this exact authenticated read before any credential or client is constructed.

> No Azure request has been made by this cell.


### 2.7 Execute the approved read-only inspection

The owner has approved exactly the request shown above. This cell is the first one in component 2 that may contact Foundry. It uses the already-authenticated local Entra/Azure CLI credential and performs only the three approved observations. It does not install packages, write Azure resources, invoke a model, or create an agent.

The result is summarized into counts and names rather than retaining raw SDK objects. Those facts establish what the project returned at this time; they do not authorize the next component.

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential


def execute_approved_read_only_discovery(
    request: ReadOnlyCloudDiscoveryRequest,
) -> dict[str, object]:
    """Perform only the approved project/deployment/agent read."""
    if request.review_status != "pending_human_review":
        raise ValueError("the request must retain its explicit review state")

    # This is the approved external boundary: use the existing local Entra login.
    credential = DefaultAzureCredential()
    client = AIProjectClient(endpoint=request.project_endpoint, credential=credential)
    try:
        # These are list/read operations only; no create, update, delete, or invoke call appears here.
        deployments = tuple(client.deployments.list())
        prompt_agents = tuple(client.agents.list())
    finally:
        client.close()
        credential.close()

    # Keep the durable notebook result compact and avoid storing raw SDK objects.
    return {
        "target": {
            "account_name": request.account_name,
            "project_name": request.project_name,
            "endpoint": request.project_endpoint,
        },
        "deployment_names": tuple(getattr(item, "name", "<unnamed>") for item in deployments),
        "prompt_agent_names": tuple(getattr(item, "name", "<unnamed>") for item in prompt_agents),
        "deployment_count": len(deployments),
        "prompt_agent_count": len(prompt_agents),
        "operations_performed": request.operations,
        "mutations_performed": False,
        "model_invocations_performed": False,
    }


FOUNDATION_READ_RESULT = execute_approved_read_only_discovery(READ_ONLY_CLOUD_DISCOVERY_REQUEST)


### 2.8 Descriptive read-result view

This final component-2 renderer formats the actual redacted result returned by the approved read. It separates observed project state from the permissions that remain closed. A zero count means the list was empty at this read time; it does not mean a deployment or agent was created.

In [ ]:
def render_read_only_discovery_result(result: dict[str, object]) -> None:
    """Presentation only: format the approved read's redacted facts."""
    from IPython.display import Markdown, display

    deployment_names = ", ".join(result["deployment_names"]) or "none"
    prompt_agent_names = ", ".join(result["prompt_agent_names"]) or "none"
    # The renderer reports observations and explicitly repeats what did not occur.
    display(Markdown(f"""### Approved read-only Foundry result

| Observed field | Result |
|:---|:---|
| Account / project | `{result['target']['account_name']}` / `{result['target']['project_name']}` |
| Model deployments | **{result['deployment_count']}** ({deployment_names}) |
| Prompt agents | **{result['prompt_agent_count']}** ({prompt_agent_names}) |
| Azure mutations | **none** |
| Model invocations | **none** |

- [ ] Review this read result before moving to deployment-request construction.

> This is a point-in-time read-only observation. It is not deployment approval or model evidence.
"""))


render_read_only_discovery_result(FOUNDATION_READ_RESULT)


### Approved read-only Foundry result

| Observed field | Result |
|:---|:---|
| Account / project | `foundry-trainer-resource` / `foundry-trainer` |
| Model deployments | **1** (evidence-analyst-luna-v1) |
| Prompt agents | **0** (none) |
| Azure mutations | **none** |
| Model invocations | **none** |

- [ ] Review this read result before moving to deployment-request construction.

> This is a point-in-time read-only observation. It is not deployment approval or model evidence.


## 3. Build and review the model-deployment request

Component 2 established the pre-apply project state without changing it: the target was reachable and no model deployment existed at that time. Component 3 now turns that historical preflight plus the reviewed desired state into a pure deployment request record. This is the bridge between "we know what was there before apply" and "we know exactly what we asked Azure to create."

This component has three bounded substeps: construct the request, validate it against the approved configuration and read-only result, and present the exact proposed action. Substep 4—human approval—is intentionally left outside the code. No Azure deployment occurs here.

### 3.1 Construct the pure deployment request

The request contains only source-visible values already established in this notebook: the target account/project, model identity, SKU/capacity, content-filter policy, version policy, and the read-only preflight facts. Constructing it does not create an Azure SDK request object or contact the service.

Because component 4 has now applied the request, a fresh live read would include the deployment and correctly trigger the duplicate-name guard. This component therefore uses the recorded pre-apply snapshot from component 2, preserving the original evidence used to justify the request.

In [17]:
@dataclass(frozen=True)
class ModelDeploymentRequest:
    """Pure, reviewable desired action; not an Azure SDK request."""

    account_name: str
    project_name: str
    resource_group: str
    deployment_name: str
    model_name: str
    model_version: str
    model_format: str
    sku_name: str
    capacity_units: int
    content_filter_policy: str
    version_upgrade_policy: str
    spillover_deployment_name: None
    preflight_deployment_names: tuple[str, ...]
    review_status: str


def build_model_deployment_request(
    config: EvidenceAnalystDeploymentConfig,
    discovery_result: dict[str, object],
) -> ModelDeploymentRequest:
    """Build a pure request from reviewed configuration and actual readback facts."""
    validate_deployment_config(config)
    target = discovery_result["target"]
    if target["account_name"] != config.account_name or target["project_name"] != config.project_name:
        raise ValueError("discovery target does not match the deployment configuration")

    # The read-only result prevents us from preparing a duplicate deployment name.
    existing_names = tuple(discovery_result["deployment_names"])
    if config.deployment_name in existing_names:
        raise ValueError("the proposed deployment name already exists")

    return ModelDeploymentRequest(
        account_name=config.account_name,
        project_name=config.project_name,
        resource_group=config.resource_group,
        deployment_name=config.deployment_name,
        model_name=config.model_name,
        model_version=config.model_version,
        model_format=config.model_format,
        sku_name=config.sku_name,
        capacity_units=config.capacity_units,
        content_filter_policy=config.content_filter_policy,
        version_upgrade_policy=config.version_upgrade_policy,
        spillover_deployment_name=config.spillover_deployment_name,
        preflight_deployment_names=existing_names,
        review_status="pending_human_review",
    )


# This is the actual component-2 result captured before the approved apply.
PRE_APPLY_DISCOVERY_RESULT = {
    "target": {
        "account_name": "foundry-trainer-resource",
        "project_name": "foundry-trainer",
        "endpoint": "https://foundry-trainer-resource.services.ai.azure.com/api/projects/foundry-trainer",
    },
    "deployment_names": (),
    "prompt_agent_names": (),
    "deployment_count": 0,
    "prompt_agent_count": 0,
    "operations_performed": (
        "read project identity/reachability",
        "list existing model deployments",
        "list existing prompt agents",
    ),
    "mutations_performed": False,
    "model_invocations_performed": False,
}
MODEL_DEPLOYMENT_REQUEST = build_model_deployment_request(
    VALIDATED_DEPLOYMENT_CONFIG,
    PRE_APPLY_DISCOVERY_RESULT,
)


ModelDeploymentRequest(account_name='foundry-trainer-resource', project_name='foundry-trainer', resource_group='rg-foundry-trainer-eastus2', deployment_name='evidence-analyst-luna-v1', model_name='gpt-5.6-luna', model_version='2026-07-09', model_format='OpenAI', sku_name='DataZoneStandard', capacity_units=1, content_filter_policy='Microsoft.DefaultV2', version_upgrade_policy='NoAutoUpgrade', spillover_deployment_name=None, preflight_deployment_names=(), review_status='pending_human_review')\n

### 3.2 Validate the request

Validation confirms that the request still matches the reviewed low-cost model envelope, the approved project target, and the actual read-only discovery result. A successful validation means the request is internally consistent; it does not mean Azure has accepted or applied it.

In [18]:
def validate_model_deployment_request(
    request: ModelDeploymentRequest,
) -> ModelDeploymentRequest:
    """Fail closed if the pure request has drifted from the reviewed envelope."""
    if request.review_status != "pending_human_review":
        raise ValueError("deployment request cannot bypass human review")
    if request.sku_name != "DataZoneStandard" or request.capacity_units != 1:
        raise ValueError("deployment request exceeds the reviewed capacity envelope")
    if request.version_upgrade_policy != "NoAutoUpgrade" or request.spillover_deployment_name is not None:
        raise ValueError("automatic version changes and spillover remain prohibited")
    if request.preflight_deployment_names:
        raise ValueError("deployment preflight is not empty; stop before creating anything")
    return request


VALIDATED_MODEL_DEPLOYMENT_REQUEST = validate_model_deployment_request(
    MODEL_DEPLOYMENT_REQUEST
)


ModelDeploymentRequest(account_name='foundry-trainer-resource', project_name='foundry-trainer', resource_group='rg-foundry-trainer-eastus2', deployment_name='evidence-analyst-luna-v1', model_name='gpt-5.6-luna', model_version='2026-07-09', model_format='OpenAI', sku_name='DataZoneStandard', capacity_units=1, content_filter_policy='Microsoft.DefaultV2', version_upgrade_policy='NoAutoUpgrade', spillover_deployment_name=None, preflight_deployment_names=(), review_status='pending_human_review')\n

### 3.3 Pretty-print the proposed action

This review card makes the exact request legible before any apply operation. It identifies the proposed Azure action, records the empty preflight, and repeats the protections that remain in force. Substep 3.4—the owner’s approval—is intentionally represented by an unchecked task rather than code.

In [19]:
def render_model_deployment_request(request: ModelDeploymentRequest) -> None:
    """Presentation only: format the validated pure request."""
    from IPython.display import Markdown, display

    # This card describes a possible action; it never calls an Azure client.
    display(Markdown(f"""### Proposed model-deployment request

| Setting | Proposed value |
|:---|:---|
| Account / project | `{request.account_name}` / `{request.project_name}` |
| Resource group | `{request.resource_group}` |
| Deployment | `{request.deployment_name}` |
| Model | `{request.model_name}` `{request.model_version}` (`{request.model_format}`) |
| SKU / capacity | `{request.sku_name}` / `{request.capacity_units}` unit = **1,000 TPM** |
| Content filter | `{request.content_filter_policy}` |
| Version upgrades | `{request.version_upgrade_policy}` |
| Spillover | none |
| Preflight deployments | **{len(request.preflight_deployment_names)}** |
| Azure action | **not performed** |
| Review state | **{request.review_status}** |

- [ ] 3.4 Approve, revise, or reject this exact deployment request.

> Validation proves local consistency only. It does not deploy a model or reserve capacity.
"""))


render_model_deployment_request(VALIDATED_MODEL_DEPLOYMENT_REQUEST)


### Proposed model-deployment request

| Setting | Proposed value |
|:---|:---|
| Account / project | `foundry-trainer-resource` / `foundry-trainer` |
| Resource group | `rg-foundry-trainer-eastus2` |
| Deployment | `evidence-analyst-luna-v1` |
| Model | `gpt-5.6-luna` `2026-07-09` (`OpenAI`) |
| SKU / capacity | `DataZoneStandard` / `1` unit = **1,000 TPM** |
| Content filter | `Microsoft.DefaultV2` |
| Version upgrades | `NoAutoUpgrade` |
| Spillover | none |
| Preflight deployments | **0** |
| Azure action | **not performed** |
| Review state | **pending_human_review** |

- [ ] 3.4 Approve, revise, or reject this exact deployment request.

> Validation proves local consistency only. It does not deploy a model or reserve capacity.


## 4. Apply and verify the approved model deployment

The owner approved the exact request above. The deployment was created against the named account and resource group using the Azure management API, then read back immediately. This section records the actual result, not a prediction.

The apply boundary was limited to one model deployment. It did not create a prompt agent, invoke a model, add tools, or change the evidence analyst policy. The next component remains a separate reviewable prompt-agent definition.

In [ ]:
# Recorded readback from the approved deployment apply; this is not a rerun command.
DEPLOYMENT_APPLY_READBACK = {
    "name": "evidence-analyst-luna-v1",
    "provisioningState": "Succeeded",
    "model": {"format": "OpenAI", "name": "gpt-5.6-luna", "version": "2026-07-09"},
    "sku": {"capacity": 1, "name": "DataZoneStandard"},
    "raiPolicyName": "Microsoft.DefaultV2",
    "versionUpgradeOption": "NoAutoUpgrade",
    "currentCapacity": 1,
}


### Descriptive deployment verification

The following card formats the actual readback. `Succeeded` confirms the resource provider completed this deployment request; it does not mean the prompt agent exists or that the model has been invoked.

In [ ]:
from IPython.display import Markdown, display

display(Markdown("""### Model deployment verification

| Verified field | Actual result |
|:---|:---|
| Deployment | `evidence-analyst-luna-v1` |
| Provisioning state | **Succeeded** |
| Model | `gpt-5.6-luna` `2026-07-09` |
| SKU / capacity | `DataZoneStandard` / **1** (1,000 TPM) |
| RAI policy | `Microsoft.DefaultV2` |
| Version upgrades | `NoAutoUpgrade` |
| Prompt agent | **not created** |
| Model invocation | **not performed** |

- [ ] Review the deployment verification before beginning component 5.

> This card reports the actual deployment readback and does not authorize agent creation.
"""))


### Model deployment verification

| Verified field | Actual result |
|:---|:---|
| Deployment | `evidence-analyst-luna-v1` |
| Provisioning state | **Succeeded** |
| Model | `gpt-5.6-luna` `2026-07-09` |
| SKU / capacity | `DataZoneStandard` / **1** (1,000 TPM) |
| RAI policy | `Microsoft.DefaultV2` |
| Version upgrades | `NoAutoUpgrade` |
| Prompt agent | **not created** |
| Model invocation | **not performed** |

- [ ] Review the deployment verification before beginning component 5.

> This card reports the actual deployment readback and does not authorize agent creation.
